In [ ]:
# ==========================================
# 1. IMPORTS & HELPER FUNCTIONS
# ==========================================
import getpass
import os
import re
import requests
from requests.auth import HTTPBasicAuth

def get_gnd_record_by_ppn(ppn: str, output_dir: str, filename: str, url: str, username: str, password: str):
    """Fetches a GND record via SRU searchRetrieve by PPN and saves the clean MARC21-xml record."""
    os.makedirs(output_dir, exist_ok=True)
    file_path = os.path.join(output_dir, filename)

    query = f'idn="{ppn}"'
    params = {
        "version": "1.1",
        "operation": "searchRetrieve",
        "query": query,
        "recordSchema": "MARC21-xml",
        "maximumRecords": "1",
    }

    print(f"Fetching original record for PPN {ppn}...")

    try:
        response = requests.get(
            url,
            params=params,
            auth=HTTPBasicAuth(username, password),
            timeout=30
        )

        if response.status_code == 200:
            raw_text = response.text

            # Isolate <record> block
            record_match = re.search(r'<record xmlns="http://www.loc.gov/MARC21/slim".*?</record>', raw_text, re.DOTALL)

            if record_match:
                record_content = record_match.group(0)

                # Ensure type="Authority"
                if 'type="Authority"' not in record_content:
                    record_content = record_content.replace(
                        '<record xmlns="http://www.loc.gov/MARC21/slim">',
                        '<record xmlns="http://www.loc.gov/MARC21/slim" type="Authority">'
                    )

                # Normalize to Unix newlines
                record_content = record_content.replace("\r\n", "\n").replace("\r", "\n")

                with open(file_path, "w", encoding="utf-8", newline="\n") as f:
                    f.write(record_content)

                print(f"[+] Success! Saved record to: {file_path}")
                return True
            else:
                print(f"[-] Could not find <record> block in SRU response for PPN {ppn}.")
                return False
        else:
            print(f"[-] Server Error during retrieval (Status {response.status_code}).")
            return False

    except Exception as e:
        print(f"[-] Connection or file operation failed: {e}")
        return False


# ==========================================
# 2. CONFIGURATION & EXECUTION
# ==========================================
SRU_READ_URL = "https://devel.dnb.de/sru/cbs-appr"

# Output settings
OUTPUT_DIR = "gnd_edit_templates"
TARGET_PPN = "950200166"
OUTPUT_FILENAME = f"ppn_{TARGET_PPN}_clean.xml"

# Credentials prompt (or from environment variables)
USERNAME_READ = os.getenv("SRU_READ_USERNAME") or input("Enter SRU Read Username: ")
PASSWORD_READ = os.getenv("SRU_READ_PASSWORD") or getpass.getpass("Enter SRU Read Password: ")

# Execute retrieval
get_gnd_record_by_ppn(
    ppn=TARGET_PPN,
    output_dir=OUTPUT_DIR,
    filename=OUTPUT_FILENAME,
    url=SRU_READ_URL,
    username=USERNAME_READ,
    password=PASSWORD_READ
)
